# Experimentos de GrantMatch registrados en MLflow

Una corrida por cada modelo de la Semana 10, sobre la preparación y la división por fecha de `pipeline_proyecto.py`. Cada corrida guarda los hiperparámetros, la fecha de corte de la división, la huella de `obt_features.csv`, el error absoluto medio en log10 en entrenamiento y prueba, los segundos de entrenamiento y el pipeline entrenado como artefacto.

La dirección del servidor sale de `MLFLOW_TRACKING_URI`; sin esa variable, MLflow escribe en `mlruns/` al lado del notebook.

In [1]:
import hashlib
import logging
import subprocess
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

# Preparacion y division del proyecto (corte por fecha en 2020-01-01).
from pipeline_proyecto import CORTE, X_ent, X_prueba, y_ent, y_prueba, preparacion

logging.getLogger("mlflow").setLevel(logging.ERROR)

RUTA_DATOS = "../../semana-09-sesion-01/plantilla-features/obt_features.csv"


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Huella de obt_features.csv:", huella_datos, "  commit:", commit)
print("Registro de MLflow:", mlflow.get_tracking_uri())
EXPERIMENTO = "grantmatch"
mlflow.set_experiment(EXPERIMENTO)

Corte: 2020-01-01. Entrenamiento: 1988-08-15 a 2019-10-01; prueba: 2020-01-01 a 2025-08-01
Entrenamiento: (123, 21) Prueba: (46, 21)
['numericas__duracion_anios', 'numericas__institutions_count', 'categoricas__escala_financiamiento_centro', 'categoricas__escala_financiamiento_individual', 'categoricas__escala_financiamiento_proyecto', 'categoricas__funder_name_Agency for Healthcare Research and Quality', 'categoricas__funder_name_American Heart Association', 'categoricas__funder_name_Bill and Melinda Gates Foundation', 'categoricas__funder_name_Cancer Prevention and Research Institute of Texas', 'categoricas__funder_name_Gordon and Betty Moore Foundation', 'categoricas__funder_name_Institute of Education Sciences', 'categoricas__funder_name_John Templeton Foundation', 'categoricas__funder_name_National Aeronautics and Space Administration', 'categoricas__funder_name_National Institute of Justice', 'categoricas__funder_name_National Institutes of Health', 'categoricas__funder_name_Natio

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1794165380358, experiment_id='3', last_update_time=1794165380358, lifecycle_stage='active', name='grantmatch', tags={}>

## Registrar una corrida por modelo

In [2]:
def registrar(nombre, modelo):
    pipeline = Pipeline([("preparar", clone(preparacion)), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({
            **modelo.get_params(),
            "fecha_corte": str(CORTE.date()),
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba),
        })
        mlflow.set_tags({"commit": commit, "mlflow.source.name": "experimentos_proyecto.ipynb"})
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metrics({
            "mae_log_entrenamiento": mae_ent,
            "mae_log_prueba": mae_prueba,
            "factor_prueba": 10 ** mae_prueba,
            "segundos": segundos,
        })
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre:32s} entrenamiento {mae_ent:.4f}   prueba {mae_prueba:.4f}   (factor {10 ** mae_prueba:.2f}x)")


registrar("linea base (mediana)", DummyRegressor(strategy="median"))
registrar("lineal", LinearRegression())
registrar("ridge alpha=1", Ridge(alpha=1.0))
registrar("ridge alpha=10", Ridge(alpha=10.0))
registrar("bosque aleatorio", RandomForestRegressor(n_estimators=200, random_state=42))
registrar("bosque min_samples_leaf=5", RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42))
registrar("gradient boosting", GradientBoostingRegressor(random_state=42))
registrar("gradient boosting regularizado",
          GradientBoostingRegressor(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=42))
registrar("red neuronal", MLPRegressor(hidden_layer_sizes=(16,), alpha=1.0, solver="lbfgs", max_iter=2000, random_state=42))

linea base (mediana)             entrenamiento 0.5158   prueba 0.3690   (factor 2.34x)


lineal                           entrenamiento 0.3206   prueba 0.3967   (factor 2.49x)


ridge alpha=1                    entrenamiento 0.3431   prueba 0.3832   (factor 2.42x)


ridge alpha=10                   entrenamiento 0.3936   prueba 0.3608   (factor 2.30x)


bosque aleatorio                 entrenamiento 0.1786   prueba 0.4233   (factor 2.65x)


bosque min_samples_leaf=5        entrenamiento 0.3629   prueba 0.3899   (factor 2.45x)


gradient boosting                entrenamiento 0.1749   prueba 0.4101   (factor 2.57x)


gradient boosting regularizado   entrenamiento 0.2810   prueba 0.3742   (factor 2.37x)


red neuronal                     entrenamiento 0.2592   prueba 0.3389   (factor 2.18x)


## Lo que quedó registrado

La regla "mediana por funder" de la Semana 10 no es un estimador de scikit-learn y no tiene pipeline que guardar; queda fuera del registro (su error en prueba era 0.37).

In [3]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_log_prueba ASC"])
columnas = ["run_id", "tags.mlflow.runName", "params.fecha_corte", "params.huella_datos",
            "metrics.mae_log_entrenamiento", "metrics.mae_log_prueba", "metrics.factor_prueba", "metrics.segundos"]
print(corridas[columnas].to_string(index=False))

                          run_id            tags.mlflow.runName params.fecha_corte params.huella_datos  metrics.mae_log_entrenamiento  metrics.mae_log_prueba  metrics.factor_prueba  metrics.segundos
669071bd3abd4ee2b1d42c880ee7bc9f                   red neuronal         2020-01-01        7503142bffe6                       0.259182                0.338908               2.182265          0.064499
505f6532318444f3a3fc7a1f03785738                 ridge alpha=10         2020-01-01        7503142bffe6                       0.393597                0.360794               2.295057          0.003581
5a528ebedd264b218b918de3014f33eb           linea base (mediana)         2020-01-01        7503142bffe6                       0.515840                0.369047               2.339090          0.003535
4432ce8eaf9d454a8c91e25ccdeaaa33 gradient boosting regularizado         2020-01-01        7503142bffe6                       0.280970                0.374169               2.366839          0.015089
b07ef

## El candidato final

La red tiene el menor error en prueba, pero la ventaja sobre Ridge (0.339 contra 0.361) son 46 awards, y la Semana 10 dejó como modelo a usar uno explicable. El candidato es **ridge alpha=10**: le pongo la etiqueta `candidato` = `final`, lo cargo desde el registro y compruebo que reproduce su número.

In [4]:
final = corridas[corridas["tags.mlflow.runName"] == "ridge alpha=10"].iloc[0]
run_final = final["run_id"]
mlflow.MlflowClient().set_tag(run_final, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + run_final + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
print("Candidato final:", final["tags.mlflow.runName"], run_final)
print(f"MAE log registrado {final['metrics.mae_log_prueba']:.6f}   reproducido {mae_reproducido:.6f}")
print("Reproduce:", abs(mae_reproducido - final["metrics.mae_log_prueba"]) < 1e-9)

# El registro exportado, ya con la etiqueta del candidato final.
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_log_prueba ASC"])
corridas.to_csv("corridas_proyecto.csv", index=False)
print(len(corridas), "corridas en corridas_proyecto.csv")

Candidato final: ridge alpha=10 505f6532318444f3a3fc7a1f03785738
MAE log registrado 0.360794   reproducido 0.360794
Reproduce: True
9 corridas en corridas_proyecto.csv
